# 3a. DTPM + SVM (Backward Compatible)

This notebook keeps the original DTPM + SVM pipeline, separated from DCNN training.
Use this when you already have trained DCNN checkpoints and want the legacy section-3 flow.

In [ ]:
import torch
import torch.nn as nn
from collections import defaultdict
from torchvision import models, transforms
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import numpy as np
import os

# set seed
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
CURR_DIR = os.getcwd()

# Update these two values before running.
DATASET = "processed_emodb_speaker_norm_loso_aug"
SPLIT_MODE = "loso"  # "original" or "loso"

DATASET_PATH = os.path.join(CURR_DIR, DATASET)
MODEL_DIR = os.path.join(CURR_DIR, "models")
RESULTS_DIR = os.path.join(CURR_DIR, "results", DATASET)
os.makedirs(RESULTS_DIR, exist_ok=True)

DCNN_PATH = os.path.join(MODEL_DIR, f"dcnn_{DATASET}.pth")
SVM_PATH = os.path.join(MODEL_DIR, f"svm_{DATASET}.joblib")

if SPLIT_MODE == "loso":
    FOLD_NAMES = sorted([
        n for n in os.listdir(DATASET_PATH)
        if n.startswith("fold") and os.path.isdir(os.path.join(DATASET_PATH, n))
    ])
else:
    FOLD_NAMES = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((227, 227), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"DATASET_PATH: {DATASET_PATH}")
print(f"SPLIT_MODE: {SPLIT_MODE}")
if SPLIT_MODE == "loso":
    print(f"Folds: {FOLD_NAMES}")

In [ ]:
# ==========================================
# 1. SETUP THE FEATURE EXTRACTOR (DCNN -> FC7)
# ==========================================
print("Preparing FC7 feature extraction utilities...")

# Make this section runnable even if Section 2 wasn't executed in this kernel.
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device was not defined. Using device: {device}")

def build_fc7_extractor(model_path):
    """Load a trained DCNN checkpoint and expose FC7 features."""
    feature_extractor = models.alexnet()
    num_ftrs = feature_extractor.classifier[6].in_features
    feature_extractor.classifier[6] = nn.Linear(num_ftrs, 7)
    feature_extractor.load_state_dict(torch.load(model_path, map_location=device))

    # Keep layers up to FC7, remove final classification layer
    feature_extractor.classifier = nn.Sequential(*list(feature_extractor.classifier.children())[:5])
    feature_extractor.eval()
    return feature_extractor.to(device)

def extract_fc7(segments_array, feature_extractor):
    """Extract FC7 features for all segments of one utterance."""
    processed_tensors = []

    for img in segments_array:
        img_copy = img.copy()

        # Min-Max scaling per channel
        for c in range(3):
            min_v, max_v = img_copy[c].min(), img_copy[c].max()
            if max_v > min_v:
                img_copy[c] = (img_copy[c] - min_v) / (max_v - min_v)
            else:
                img_copy[c] = 0.0

        img_transposed = img_copy.transpose(1, 2, 0)
        # ToTensor may keep float64 when source ndarray is float64; force float32 for AlexNet.
        tensor = transform(img_transposed).float().unsqueeze(0)
        processed_tensors.append(tensor)

    batch_tensor = torch.cat(processed_tensors).to(device=device, dtype=torch.float32)
    with torch.no_grad():
        return feature_extractor(batch_tensor).cpu().numpy()

if SPLIT_MODE == "original":
    if not os.path.exists(DCNN_PATH):
        raise FileNotFoundError(f"DCNN checkpoint not found: {DCNN_PATH}")
    fc7_extractor = build_fc7_extractor(DCNN_PATH)
    print(f"Loaded FC7 extractor from: {DCNN_PATH}")
else:
    print("LOSO mode: FC7 extractor will be loaded per fold.")

In [ ]:
# ==========================================
# 3. DATA PROCESSING PIPELINE
# ==========================================
def process_dataset_to_global_features(dataset_dir, split_name, feature_extractor):
    """Build utterance-level DTPM features from segment-level arrays."""
    print(f"\nProcessing {split_name} set from: {dataset_dir}")
    X_segs = np.load(os.path.join(dataset_dir, f"X_{split_name}.npy"))
    y_lbls = np.load(os.path.join(dataset_dir, f"y_{split_name}.npy"))
    u_ids = np.load(os.path.join(dataset_dir, f"utterance_ids_{split_name}.npy"))

    # Group all segments belonging to the same utterance id.
    utterance_dict = {}
    for i, uid in enumerate(u_ids):
        if uid not in utterance_dict:
            utterance_dict[uid] = {'segments': [], 'label': y_lbls[i]}
        utterance_dict[uid]['segments'].append(X_segs[i])

    X_global, y_global = [], []
    for uid, data in utterance_dict.items():
        # 1) FC7 per segment, 2) DTPM temporal pooling, 3) store utterance label.
        fc7_feats = extract_fc7(np.array(data['segments']), feature_extractor)
        global_feat = dtpm(fc7_feats, L=2)
        X_global.append(global_feat)
        y_global.append(data['label'])

    return np.array(X_global), np.array(y_global)

print("Feature pipeline ready.")
if SPLIT_MODE == "original":
    print("Original mode: global features will be computed in the next cell.")
else:
    print("LOSO mode: global features will be computed fold-by-fold in the next cell.")

In [ ]:
# ==========================================
# 2. DISCRIMINANT TEMPORAL PYRAMID MATCHING (DTPM)
# ==========================================
def lp_norm_pooling(matrix, p=1.12):
    # Lp pooling used by DTPM to summarize each temporal block.
    return np.power(np.mean(np.power(np.abs(matrix), p), axis=0), 1/p)

def dtpm(segment_features, L=2):
    """Convert chronological FC7 segment vectors into one utterance-level vector."""
    N = segment_features.shape[0]
    final_feature = []

    # Level 0: full sequence (1 block)
    final_feature.append(lp_norm_pooling(segment_features) * (1 / (2**L)))

    # Level 1: split into 2 temporal halves
    if N >= 2:
        mid = N // 2
        final_feature.append(lp_norm_pooling(segment_features[:mid]) * (1 / (2**L)))
        final_feature.append(lp_norm_pooling(segment_features[mid:]) * (1 / (2**L)))
    else:
        final_feature.extend([np.zeros(4096)] * 2)

    # Level 2: split into 4 temporal quarters
    if L == 2:
        if N >= 4:
            q_size = N // 4
            for i in range(4):
                start = i * q_size
                end = (i + 1) * q_size if i != 3 else N
                final_feature.append(lp_norm_pooling(segment_features[start:end]) * (1 / (2**(L-1))))
        else:
            final_feature.extend([np.zeros(4096)] * 4)

    return np.concatenate(final_feature)

In [ ]:
# ==========================================
# 4. SVM CLASSIFICATION (ORIGINAL + LOSO)
# ==========================================
svm_logs = []
def log_svm(message):
    print(message)
    svm_logs.append(str(message))

# These lists hold final utterance-level labels/predictions.
svm_all_true = []
svm_all_pred = []

if SPLIT_MODE == "original":
    log_svm("Running SVM in ORIGINAL mode...")

    # Build global utterance-level features from train and test splits
    X_train_global, y_train_global = process_dataset_to_global_features(DATASET_PATH, "train", fc7_extractor)
    X_test_global, y_test_global = process_dataset_to_global_features(DATASET_PATH, "test", fc7_extractor)

    # Standardize features before linear SVM
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_global)
    X_test_scaled = scaler.transform(X_test_global)

    svm_classifier = SVC(kernel='linear', decision_function_shape='ovo')
    svm_classifier.fit(X_train_scaled, y_train_global)
    predictions = svm_classifier.predict(X_test_scaled)

    svm_all_true = y_test_global.tolist()
    svm_all_pred = predictions.tolist()

    # Save artifacts for original mode
    joblib.dump(svm_classifier, SVM_PATH)
    joblib.dump(scaler, os.path.join(MODEL_DIR, f"scaler_{DATASET}.joblib"))
    log_svm(f"Saved SVM model to: {SVM_PATH}")

else:
    log_svm("Running SVM in LOSO mode (train/evaluate per fold)...")

    for fold_name in FOLD_NAMES:
        fold_path       = os.path.join(DATASET_PATH, fold_name)
        fold_dcnn_path  = os.path.join(MODEL_DIR, f"dcnn_{DATASET}_{fold_name}.pth")

        if not os.path.exists(fold_dcnn_path):
            raise FileNotFoundError(
                f"DCNN checkpoint for {fold_name} not found: {fold_dcnn_path}. "
                "Run DCNN LOSO training section first."
            )

        log_svm("-" * 70)
        log_svm(f"Fold: {fold_name}")

        # Load fold-specific DCNN and extract fold-specific global features
        fold_extractor = build_fc7_extractor(fold_dcnn_path)
        X_train_global, y_train_global = process_dataset_to_global_features(fold_path, "train", fold_extractor)
        X_test_global, y_test_global = process_dataset_to_global_features(fold_path, "test", fold_extractor)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_global)
        X_test_scaled = scaler.transform(X_test_global)

        fold_svm = SVC(kernel='linear', decision_function_shape='ovo')
        fold_svm.fit(X_train_scaled, y_train_global)
        fold_predictions = fold_svm.predict(X_test_scaled)

        fold_acc = accuracy_score(y_test_global, fold_predictions)
        log_svm(f"Fold utterance-level SVM accuracy: {fold_acc * 100:.2f}%")

        svm_all_true.extend(y_test_global.tolist())
        svm_all_pred.extend(fold_predictions.tolist())

        # Save each fold model/scaler separately
        fold_svm_path    = os.path.join(MODEL_DIR, f"svm_{DATASET}_{fold_name}.joblib")
        fold_scaler_path = os.path.join(MODEL_DIR, f"scaler_{DATASET}_{fold_name}.joblib")
        joblib.dump(fold_svm, fold_svm_path)
        joblib.dump(scaler, fold_scaler_path)
        log_svm(f"Saved fold SVM to: {fold_svm_path}")

# Final collated SVM results (single split for original, all folds for LOSO)
svm_y_true_final    = np.array(svm_all_true)
svm_y_pred_final    = np.array(svm_all_pred)
svm_acc_final       = accuracy_score(svm_y_true_final, svm_y_pred_final)
svm_report_final    = classification_report(svm_y_true_final, svm_y_pred_final)

log_svm("=" * 70)
log_svm(f"Final SVM utterance-level accuracy ({SPLIT_MODE}): {svm_acc_final * 100:.2f}%")
log_svm("Final SVM classification report:")
print(svm_report_final)
svm_logs.append(svm_report_final)

svm_report_path = os.path.join(RESULTS_DIR, f"classification_report_{DATASET}.txt")
with open(svm_report_path, "w") as f:
    for line in svm_logs:
        f.write(f"{line}\n")
log_svm(f"Saved classification log to: {svm_report_path}")

In [ ]:
# Keep backward-compatible variable names for downstream plotting/report cells
predictions = svm_y_pred_final
y_test_global = svm_y_true_final
accuracy = svm_acc_final
report = svm_report_final

print("SVM final results are ready for plotting and saved reporting.")
print(f"Accuracy (WAR): {accuracy * 100:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from utility import EMOTION_ENG_MAP

EMOTION_NAMES = [EMOTION_ENG_MAP[i] for i in range(len(EMOTION_ENG_MAP))]
cm = confusion_matrix(y_test_global, predictions, labels=np.arange(len(EMOTION_NAMES)))

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=EMOTION_NAMES,
    yticklabels=EMOTION_NAMES
)
plt.title(f'SVM Confusion Matrix ({SPLIT_MODE.upper()})', fontsize=14, pad=15)
plt.xlabel('Predicted Emotion', fontsize=12)
plt.ylabel('True Emotion', fontsize=12)
plt.tight_layout()

svm_cm_path = os.path.join(RESULTS_DIR, f"confusion_svm_{DATASET}.png")
plt.savefig(svm_cm_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved SVM confusion matrix to: {svm_cm_path}")